# FILE 1 — Preprocessing-Variant Inference

**The question.** Your four cohorts were preprocessed by four different scripts, and they do **not** agree on the
order of CLAHE and resize:

| cohort | script | order |
|---|---|---|
| OAI | `oai_preprocess.ipynb` | `clahe.apply(crop)` → resize |
| NHANES | `nhanes3_batch_process.ipynb` | `clahe.apply(crop)` → resize |
| MRKR | `mrkr_preprocess_part1.ipynb` | resize → `clahe.apply(resized)` |
| Mendeley | `mendeley_download.ipynb` | CLAHE on Chen's already-processed image → resize |

This is not cosmetic. `tileGridSize=(8,8)` on a ~2000px DICOM crop gives tiles covering ~250px of bone; on a
224px image each tile covers ~28px. **Same call, same parameters, different operation.**

**The experiment.** Take OAI's raw DICOMs — the one cohort where you still have the source — and push the *same
knee* through each pipeline order. Same anatomy, same ground-truth label, same model weights. Any difference in
the prediction is caused by **preprocessing alone**, with zero confound from patients, scanners, prevalence, or
graders.

**Variants produced**
| name | pipeline |
|---|---|
| `oai_order` | normalize → split → **CLAHE @ full-res → resize 224** (reproduces your OAI/NHANES path) |
| `mrkr_order` | normalize → split → **resize 224 → CLAHE @ 224** (reproduces your MRKR path) |
| `double_clahe` | `oai_order`, then CLAHE **again** at 224 (approximates the Mendeley double-processing) |
| `no_clahe` | normalize → split → resize 224, **no CLAHE** (control: what does CLAHE buy at all?) |

**Output:** `preprocess_variants_findings.csv` → analysed by **FILE 2**.

**Run order:** this notebook → FILE 2. Needs GPU + Drive (reads DICOMs, runs the trained checkpoint).
It does **not** retrain anything.

## Setup — same environment as the MT trainer

This mirrors `mt2_multitask_trainer.ipynb` cells 0–6 so the notebook is self-contained (same `TM` helpers, same
`MultiTaskNet` definition, same manifest). Nothing here is retrained.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib, os
sys.path.insert(0, '/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import numpy as np, pandas as pd, json, glob, time, math, re
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
import cv2
from PIL import Image
if 'training_lib_max' in sys.modules: importlib.reload(sys.modules['training_lib_max'])
import training_lib_max as TM
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda': raise RuntimeError('No GPU.')

PROJECT = Path('/content/drive/MyDrive/Master Thesis')
MT_ROOT = PROJECT/'scope3_mt'; MT_CKPT = MT_ROOT/'checkpoints'; MT_RES = MT_ROOT/'results'
MT_MANIFEST = MT_ROOT/'manifest_mt.csv'
assert MT_MANIFEST.exists(), 'Run MT-1 first to create manifest_mt.csv'
mt_man = pd.read_csv(str(MT_MANIFEST))
SUB  = [c for c in ['osteophyte_max','jsn_max','sclerosis_max'] if c in mt_man.columns]
MASK = [c.replace('_max','_mask') for c in SUB]
print('Sub-features:', SUB)

Mounted at /content/drive
Sub-features: ['osteophyte_max', 'jsn_max', 'sclerosis_max']


In [ ]:
SUBK = 4
class MultiTaskNet(nn.Module):
    def __init__(self, n_sub):
        super().__init__()
        self.core = TM.OrdinalNet(config.NUM_CLASSES, 4, use_hierarchical=True)
        feat = self.core.feat_dim
        self.sub_heads = nn.ModuleList([nn.Sequential(nn.Flatten(1), nn.LayerNorm(feat), nn.Dropout(0.3),
                                                      nn.Linear(feat, SUBK-1)) for _ in range(n_sub)])
    def forward(self, x, grl_lambda=0.0):
        f = self.core.backbone(x)
        if f.dim() == 4: f = f.mean(dim=[-2,-1])
        kl = self.core.corn(f); s1 = self.core.head_s1(f); s2 = self.core.head_s2(f)
        dom = self.core.domain_head(TM.grad_reverse(f, grl_lambda))
        subs = [h(f) for h in self.sub_heads]
        return kl, s1, s2, dom, subs
print('MultiTaskNet defined.')

MultiTaskNet defined.


In [ ]:
# ------------------------------- CONFIG -------------------------------
# Which trained fold to use. IMPORTANT: do NOT use mt_oai_seed0 -- OARSI sub-grades exist only for
# OAI, so the fold that holds OAI out has NO sub-feature supervision and its findings heads are junk.
RUN_NAME  = 'mt_mendeley_seed0'
N_KNEES   = 1500        # sample size (None = all matched OAI knees). 1500 is plenty for this effect.
BATCH     = 16
SEED      = 0
OUT_CSV   = MT_RES/'preprocess_variants_findings.csv'

IMAGE_DIR  = PROJECT/'oai'/'screening_images'
LABELS_CSV = PROJECT/'oai'/'oai_labels.csv'
IMG_SIZE   = 224
CLAHE_CLIP = 2.0
CLAHE_TILE = (8, 8)
PREFERRED_SERIES = '0.C.2'
VARIANTS = ['oai_order', 'mrkr_order', 'double_clahe', 'no_clahe']
print('using checkpoint:', RUN_NAME, '| variants:', VARIANTS)

using checkpoint: mt_mendeley_seed0 | variants: ['oai_order', 'mrkr_order', 'double_clahe', 'no_clahe']


## The four pipelines

Copied verbatim in spirit from your own scripts — `load_dicom_array` and `split_bilateral` are byte-for-byte the
OAI logic, so the *only* thing that varies between variants is where CLAHE sits relative to the resize.

In [ ]:
clahe_proc = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_TILE)
!pip install pydicom
import pydicom

def load_dicom_array(path):
    # identical to oai_preprocess.ipynb
    ds  = pydicom.dcmread(str(path))
    arr = ds.pixel_array.astype(np.float32)
    arr = arr * float(getattr(ds, 'RescaleSlope', 1)) + float(getattr(ds, 'RescaleIntercept', 0))
    if getattr(ds, 'PhotometricInterpretation', 'MONOCHROME2') == 'MONOCHROME1':
        arr = arr.max() - arr
    lo, hi = float(np.percentile(arr, 1)), float(np.percentile(arr, 99))
    if hi - lo < 1: lo, hi = float(arr.min()), float(arr.max())
    if hi - lo < 1: return np.zeros(arr.shape, dtype=np.uint8)
    return np.clip((arr - lo) / (hi - lo) * 255.0, 0, 255).astype(np.uint8)

def split_bilateral(arr):
    mid = arr.shape[1] // 2
    return arr[:, :mid], np.fliplr(arr[:, mid:])

def _resize_lanczos(a, size=IMG_SIZE):
    return np.array(Image.fromarray(a, 'L').resize((size, size), Image.LANCZOS))

def make_variant(crop, variant):
    if variant == 'oai_order':       # CLAHE at full DICOM resolution, then downsize
        return _resize_lanczos(clahe_proc.apply(crop))
    if variant == 'mrkr_order':      # downsize first, then CLAHE at 224
        return clahe_proc.apply(cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LANCZOS4))
    if variant == 'double_clahe':    # OAI order, then CLAHE a second time at 224 (Mendeley-like)
        return clahe_proc.apply(_resize_lanczos(clahe_proc.apply(crop)))
    if variant == 'no_clahe':        # control
        return _resize_lanczos(crop)
    raise ValueError(variant)
print('pipelines defined:', VARIANTS)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.9 MB/s eta 0:00:00
pipelines defined: ['oai_order', 'mrkr_order', 'double_clahe', 'no_clahe']


In [ ]:
print('Scanning Drive for OAI DICOMs (may take a minute) ...')
subject_dicoms = {}
for p in IMAGE_DIR.rglob('*'):
    if p.is_file() and p.suffix == '':
        parts = p.relative_to(IMAGE_DIR).parts
        if len(parts) >= 2:
            subject_dicoms.setdefault(parts[1], {})[parts[0]] = p

def best_dicom(subject):
    sm = subject_dicoms.get(subject, {})
    return (sm.get(PREFERRED_SERIES) or next(iter(sm.values()))) if sm else None

labels = pd.read_csv(LABELS_CSV, dtype={'subject_id': str})
labels['subject_id'] = labels['subject_id'].str.strip()
labels = labels[labels.subject_id.isin(subject_dicoms.keys())].reset_index(drop=True)
print('matched knee rows with a DICOM:', len(labels))

# attach OARSI ground truth (only OAI has it) so FILE 2 can run intervention on these rows
mt_oai = mt_man[mt_man.dataset == 'oai'].copy()
mt_oai['subject_id'] = mt_oai.filename.astype(str).str.extract(r'OAI_(\d{7})_')[0]
mt_oai['side']       = mt_oai.filename.astype(str).str.extract(r'OAI_\d{7}_([LR])_')[0]
gt = mt_oai.set_index(['subject_id','side'])[SUB]

if N_KNEES:
    labels = labels.sample(n=min(N_KNEES, len(labels)), random_state=SEED).reset_index(drop=True)
print('knees to process:', len(labels), '| x', len(VARIANTS), 'variants =',
      len(labels)*len(VARIANTS), 'forward passes')

Scanning Drive for OAI DICOMs (may take a minute) ...
matched knee rows with a DICOM: 8921
knees to process: 1500 | x 4 variants = 6000 forward passes


In [ ]:
ckb = MT_CKPT/f'{RUN_NAME}_best.pt'
assert ckb.exists(), f'{ckb} not found -- train that fold first'
model = MultiTaskNet(len(SUB)).to(device)
TM.load_ckpt(str(ckb), model, None)
model.eval()
print('reloaded ->', ckb)

@torch.no_grad()
def predict_batch(imgs224):
    # feed each variant through the SAME downstream path the model was trained with,
    # so the only difference between variants is the CLAHE/resize order upstream
    batch = []
    for a in imgs224:
        a = TM._resize(TM.joint_crop(a))
        x = torch.from_numpy(a.astype(np.float32)/255.0)
        x = (x - 0.485)/0.229
        batch.append(x.unsqueeze(0).repeat(3,1,1))
    xb = torch.stack(batch).to(device)
    kl, _, _, _, subs = model(xb, grl_lambda=0.0)
    return TM.corn_probs(kl).cpu().numpy(), [TM.corn_probs(s).cpu().numpy() for s in subs]

Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth


100%|██████████| 755M/755M [00:03<00:00, 230MB/s]


reloaded -> /content/drive/MyDrive/Master Thesis/scope3_mt/checkpoints/mt_mendeley_seed0_best.pt


In [ ]:
import time as _time
rows = []
t0 = _time.time(); n_err = 0
groups = list(labels.groupby('subject_id'))
print('processing %d subjects ...' % len(groups))

for gi, (subject_id, grp) in enumerate(groups):
    dp = best_dicom(subject_id)
    if dp is None: n_err += len(grp); continue
    try:
        full = load_dicom_array(dp)
        right, left = split_bilateral(full)
    except Exception:
        n_err += len(grp); continue
    crops = {'R': right, 'L': left}
    for _, r in grp.iterrows():
        side = str(r['side']).strip().upper()[:1]
        if side not in crops: n_err += 1; continue
        crop = crops[side]
        rec = {'subject_id': subject_id, 'side': side,
               'knee_key': f'{subject_id}{side}', 'kl_label': int(r['kl_grade'])}
        try:
            g = gt.loc[(subject_id, side)]
            for c in SUB: rec[c.replace('_max','') + '_true'] = float(g[c])
        except Exception:
            for c in SUB: rec[c.replace('_max','') + '_true'] = np.nan
        for v in VARIANTS:
            try:
                img = make_variant(crop, v)
                klp, subp = predict_batch([img])
                rec[f'{v}__kl_pred'] = int(klp[0].argmax())
                for k, c in enumerate(SUB):
                    rec[f'{v}__{c.replace("_max","")}_pred'] = int(subp[k][0].argmax())
            except Exception:
                rec[f'{v}__kl_pred'] = np.nan
        rows.append(rec)
    if (gi+1) % 50 == 0:
        el = _time.time()-t0
        print('  [%d/%d subjects] %d knees | %.1fs | ETA %.0fs' %
              (gi+1, len(groups), len(rows), el, el/(gi+1)*(len(groups)-gi-1)))

out = pd.DataFrame(rows)
out.to_csv(OUT_CSV, index=False)
print('\nsaved ->', OUT_CSV, '| knees:', len(out), '| errors:', n_err)
print(out.head())

processing 1371 subjects ...


/tmp/ipykernel_1908/2387978723.py:22: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return np.array(Image.fromarray(a, 'L').resize((size, size), Image.LANCZOS))


  [50/1371 subjects] 54 knees | 56.2s | ETA 1486s
  [100/1371 subjects] 108 knees | 114.2s | ETA 1452s
  [150/1371 subjects] 162 knees | 172.6s | ETA 1405s
  [200/1371 subjects] 214 knees | 228.5s | ETA 1338s
  [250/1371 subjects] 265 knees | 279.7s | ETA 1254s
  [300/1371 subjects] 317 knees | 335.4s | ETA 1198s
  [350/1371 subjects] 368 knees | 385.1s | ETA 1123s
  [400/1371 subjects] 416 knees | 437.2s | ETA 1061s
  [450/1371 subjects] 469 knees | 498.9s | ETA 1021s


/usr/local/lib/python3.12/dist-packages/pydicom/charset.py:727: UserWarning: Incorrect value for Specific Character Set 'ISO_2022_IR_6' - assuming 'ISO 2022 IR 6'
  _warn_about_invalid_encoding(encoding, patched)


  [500/1371 subjects] 523 knees | 555.4s | ETA 967s


...

  [1100/1371 subjects] 1156 knees | 1198.1s | ETA 295s


  [1150/1371 subjects] 1211 knees | 1248.9s | ETA 240s


  [1200/1371 subjects] 1261 knees | 1299.6s | ETA 185s


  [1250/1371 subjects] 1315 knees | 1351.3s | ETA 131s


  [1300/1371 subjects] 1369 knees | 1401.9s | ETA 77s
  [1350/1371 subjects] 1422 knees | 1455.6s | ETA 23s

saved -> /content/drive/MyDrive/Master Thesis/scope3_mt/results/preprocess_variants_findings.csv | knees: 1444 | errors: 56
  subject_id side  knee_key  kl_label  osteophyte_true  jsn_true  \
0    9001104    R  9001104R         3              2.0       2.0   
1    9001897    R  9001897R         3              1.0       2.0   
2    9001897    L  9001897L         0              0.0       0.0   
3    9002411    L  9002411L         2              1.0       1.0   
4    9002430    L  9002430L         3              1.0       2.0   

   sclerosis_true  oai_order__kl_pred  oai_order__osteophyte_pred  \
0             2.0                   3                           1   
1             2.0                   3                           1   
2             0.0                   2                           0   
3             1.0                   2                           1   
4            